In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install torch torchvision torchaudio -q
!pip install torch-geometric -q

In [4]:

# ================================================================
# CONVNEXT-TINY + GCN FOR FISH4KNOWLEDGE DATASET
# 23-Class Fish Classification
# Stratified 64:16:20 Train / Validation / Test
# Bootstrap 95% Confidence Intervals
# ================================================================

import os
import random
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image, ImageFile
from tqdm.auto import tqdm

from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ================================================================
# 1. CONFIGURATION
# ================================================================

ROOT_DIR = Path(
    "/kaggle/input/datasets/mddatamanic/rahman-fish-dataset/Rahman Image dataset"
)

OUTPUT_DIR = Path("/kaggle/working/Fish4Knowledge_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4

HIDDEN_DIM = 128
DROPOUT = 0.3

SEED = 42
NUM_BOOT = 2000
ALPHA = 0.05

NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("=" * 65)
print("CONVNEXT-TINY + GCN FISH CLASSIFICATION")
print("=" * 65)
print("Device:", device)

# ================================================================
# 2. DATASET PATH VALIDATION
# ================================================================

if not ROOT_DIR.exists():

    print("\nSpecified dataset path does not exist.")
    print("Searching Kaggle input directories...")

    candidates = []

    for root, dirs, files in os.walk("/kaggle/input"):

        class_dirs = [
            d for d in dirs
            if d[:2].isdigit() and "_" in d
        ]

        if len(class_dirs) >= 10:
            candidates.append(Path(root))

    if candidates:
        print("\nPossible dataset directories:")
        for p in candidates[:10]:
            print(p)

        raise FileNotFoundError(
            "Please update ROOT_DIR using one of the paths above."
        )

    raise FileNotFoundError(
        "Dataset directory not found. Check the Kaggle dataset attachment."
    )

# ================================================================
# 3. ROBUST IMAGE DISCOVERY
# ================================================================

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".webp", ".tif", ".tiff"
}

classes = sorted([
    d.name for d in ROOT_DIR.iterdir()
    if d.is_dir()
])

if len(classes) == 0:
    raise RuntimeError(
        f"No class folders found inside {ROOT_DIR}"
    )

image_paths = []
labels = []
class_counts = {}

print("\nScanning class folders recursively...\n")

for class_idx, class_name in enumerate(classes):

    class_dir = ROOT_DIR / class_name

    files = [
        p for p in class_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    class_counts[class_name] = len(files)

    for img_path in files:
        image_paths.append(str(img_path))
        labels.append(class_idx)

    print(f"{class_name}: {len(files)} images")

labels = np.asarray(labels, dtype=np.int64)

num_classes = len(classes)

print("\n" + "=" * 65)
print("DATASET SUMMARY")
print("=" * 65)

print("Number of classes:", num_classes)
print("Total images:", len(image_paths))

if len(image_paths) == 0:

    raise RuntimeError(
        "ZERO IMAGES FOUND. The class directories exist, but "
        "the image files were not found. Inspect the actual "
        "folder structure and file extensions."
    )

empty_classes = [
    name for name, count in class_counts.items()
    if count == 0
]

if empty_classes:
    raise RuntimeError(
        f"Empty class folders detected: {empty_classes}"
    )

if num_classes != 23:
    print("WARNING: Expected 23 classes, found:", num_classes)

if np.min(np.bincount(labels)) < 5:
    raise RuntimeError(
        "At least one class has fewer than 5 images. "
        "A stratified train/validation/test split may not be reliable."
    )

# Save class distribution
distribution_df = pd.DataFrame({
    "Class": classes,
    "Number_of_Images": [
        class_counts[c] for c in classes
    ]
})

distribution_df.to_csv(
    OUTPUT_DIR / "class_distribution.csv",
    index=False
)

# ================================================================
# 4. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ================================================================

indices = np.arange(len(image_paths))

# First: 80% development, 20% independent test
trainval_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

# Second: 80% train and 20% validation from development set
# Final proportions: 64% train, 16% validation, 20% test
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=labels[trainval_idx]
)

print("\nDATA SPLIT")
print("-" * 50)
print(f"Training:   {len(train_idx)} ({len(train_idx)/len(indices)*100:.2f}%)")
print(f"Validation: {len(val_idx)} ({len(val_idx)/len(indices)*100:.2f}%)")
print(f"Testing:    {len(test_idx)} ({len(test_idx)/len(indices)*100:.2f}%)")

# ================================================================
# 5. CONVNEXT IMAGE TRANSFORM
# ================================================================

weights = ConvNeXt_Tiny_Weights.DEFAULT
transform = weights.transforms()

# ================================================================
# 6. FISH DATASET
# ================================================================

class FishDataset(Dataset):

    def __init__(self, paths, targets, transform=None):

        self.paths = paths
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        path = self.paths[idx]

        try:
            image = Image.open(path).convert("RGB")

        except Exception as e:
            raise RuntimeError(
                f"Cannot read image: {path}"
            ) from e

        if self.transform:
            image = self.transform(image)

        return image, int(self.targets[idx])


train_ds = FishDataset(
    [image_paths[i] for i in train_idx],
    labels[train_idx],
    transform
)

val_ds = FishDataset(
    [image_paths[i] for i in val_idx],
    labels[val_idx],
    transform
)

test_ds = FishDataset(
    [image_paths[i] for i in test_idx],
    labels[test_idx],
    transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

train_eval_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ================================================================
# 7. CONVNEXT-TINY FEATURE EXTRACTOR
# ================================================================

class ConvNeXtFeatureExtractor(nn.Module):

    def __init__(self):

        super().__init__()

        base = convnext_tiny(weights=weights)

        self.features = base.features
        self.avgpool = base.avgpool

        # Freeze pretrained ConvNeXt
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x):

        return self.features(x)


convnext_model = ConvNeXtFeatureExtractor().to(device)
convnext_model.eval()

with torch.no_grad():

    dummy = torch.randn(
        2, 3, 224, 224,
        device=device
    )

    feature_map = convnext_model(dummy)

C = feature_map.shape[1]
Hf = feature_map.shape[2]
Wf = feature_map.shape[3]

print("\nConvNeXt feature map:", (C, Hf, Wf))

# ================================================================
# 8. GRID GRAPH CONSTRUCTION
# ================================================================

def build_grid_edge_index(H, W):

    edges = []

    for i in range(H):

        for j in range(W):

            idx = i * W + j

            if i + 1 < H:
                edges.append((idx, idx + W))
                edges.append((idx + W, idx))

            if j + 1 < W:
                edges.append((idx, idx + 1))
                edges.append((idx + 1, idx))

    return torch.tensor(
        edges,
        dtype=torch.long
    ).t().contiguous()


EDGE_INDEX = build_grid_edge_index(Hf, Wf)


def features_to_graph_batch(features, targets):

    features = features.detach().cpu()
    targets = targets.detach().cpu()

    graphs = []

    for b in range(features.shape[0]):

        fm = features[b]

        x = fm.permute(1, 2, 0).reshape(-1, C)

        graph = Data(
            x=x,
            edge_index=EDGE_INDEX,
            y=targets[b].view(1).long()
        )

        graphs.append(graph)

    return Batch.from_data_list(graphs)


# ================================================================
# 9. GCN CLASSIFIER
# ================================================================

class GCNClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        dropout=0.3
    ):

        super().__init__()

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.lin = nn.Linear(hidden_dim, output_dim)

        self.dropout = dropout

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = global_mean_pool(x, batch)

        logits = self.lin(x)

        return logits


gcn_model = GCNClassifier(
    input_dim=C,
    hidden_dim=HIDDEN_DIM,
    output_dim=num_classes,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.AdamW(
    gcn_model.parameters(),
    lr=LR
)

criterion = nn.CrossEntropyLoss()

# ================================================================
# 10. FEATURE EXTRACTION + PREDICTION
# ================================================================

@torch.no_grad()
def predict_on_loader(loader):

    convnext_model.eval()
    gcn_model.eval()

    all_y = []
    all_pred = []
    all_prob = []

    for imgs, ys in tqdm(loader, desc="Evaluation"):

        imgs = imgs.to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)

        feats = convnext_model(imgs)

        graph_batch = features_to_graph_batch(
            feats,
            ys
        ).to(device)

        logits = gcn_model(graph_batch)

        probs = F.softmax(logits, dim=1)

        preds = probs.argmax(dim=1)

        all_y.append(ys.cpu().numpy())
        all_pred.append(preds.cpu().numpy())
        all_prob.append(probs.cpu().numpy())

    return (
        np.concatenate(all_y),
        np.concatenate(all_pred),
        np.concatenate(all_prob)
    )


# ================================================================
# 11. TRAINING
# ================================================================

best_val_acc = -1.0

history = []

best_model_path = (
    OUTPUT_DIR / "ConvNeXt_Tiny_GCN_Best.pth"
)

for epoch in range(1, EPOCHS + 1):

    gcn_model.train()
    convnext_model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS}"
    )

    for imgs, ys in progress:

        imgs = imgs.to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)

        with torch.no_grad():

            feats = convnext_model(imgs)

        graph_batch = features_to_graph_batch(
            feats,
            ys
        ).to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = gcn_model(graph_batch)

        loss = criterion(logits, ys)

        loss.backward()

        optimizer.step()

        batch_size = ys.size(0)

        total_loss += loss.item() * batch_size

        preds = logits.argmax(dim=1)

        total_correct += (preds == ys).sum().item()
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = total_loss / total_samples

    train_acc = total_correct / total_samples

    yv, pv, _ = predict_on_loader(val_loader)

    val_acc = accuracy_score(yv, pv)

    print("\n" + "-" * 55)

    print(f"Epoch: {epoch}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc*100:.2f}%")
    print(f"Validation Accuracy: {val_acc*100:.2f}%")

    history.append({
        "Epoch": epoch,
        "Train_Loss": train_loss,
        "Train_Accuracy": train_acc,
        "Validation_Accuracy": val_acc
    })

    # Save best validation model
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": gcn_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_accuracy": val_acc,
                "classes": classes
            },
            best_model_path
        )

        print("Best model checkpoint saved.")

# Save training history
pd.DataFrame(history).to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

# ================================================================
# 12. LOAD BEST MODEL
# ================================================================

checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

gcn_model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("\nBest epoch:", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["val_accuracy"])

# ================================================================
# 13. BOOTSTRAP CONFIDENCE INTERVALS
# ================================================================

def compute_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    alpha=0.05
):

    # Point estimates
    acc = accuracy_score(y_true, y_pred)

    prec = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    rec = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    auc = roc_auc_score(
        y_true,
        y_prob,
        labels=np.arange(num_classes),
        multi_class="ovr",
        average="macro"
    )

    # Stratified bootstrap:
    # Resample independently within each true class.
    # This preserves all classes in every bootstrap sample.
    rng = np.random.default_rng(SEED)

    class_indices = [
        np.where(y_true == c)[0]
        for c in range(num_classes)
    ]

    boot_metrics = {
        "Accuracy": [],
        "Precision": [],
        "Recall": [],
        "F1": [],
        "AUC": []
    }

    for _ in tqdm(
        range(n_boot),
        desc="Bootstrap confidence intervals"
    ):

        sampled_idx = np.concatenate([
            rng.choice(
                idx,
                size=len(idx),
                replace=True
            )
            for idx in class_indices
        ])

        yt = y_true[sampled_idx]
        yp = y_pred[sampled_idx]
        ypb = y_prob[sampled_idx]

        boot_metrics["Accuracy"].append(
            accuracy_score(yt, yp)
        )

        boot_metrics["Precision"].append(
            precision_score(
                yt, yp,
                average="macro",
                zero_division=0
            )
        )

        boot_metrics["Recall"].append(
            recall_score(
                yt, yp,
                average="macro",
                zero_division=0
            )
        )

        boot_metrics["F1"].append(
            f1_score(
                yt, yp,
                average="macro",
                zero_division=0
            )
        )

        boot_metrics["AUC"].append(
            roc_auc_score(
                yt,
                ypb,
                labels=np.arange(num_classes),
                multi_class="ovr",
                average="macro"
            )
        )

    point_estimates = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    }

    results = {}

    for metric, values in boot_metrics.items():

        values = np.asarray(values)

        lower = np.percentile(
            values,
            100 * alpha / 2
        )

        upper = np.percentile(
            values,
            100 * (1 - alpha / 2)
        )

        results[metric] = {
            "Point": point_estimates[metric],
            "Mean": values.mean(),
            "Lower": lower,
            "Upper": upper
        }

    return results


# ================================================================
# 14. FINAL EVALUATION
# ================================================================

splits = {
    "TRAIN": train_eval_loader,
    "VALIDATION": val_loader,
    "TEST": test_loader
}

all_results = []

test_true = None
test_pred = None
test_prob = None

for split_name, loader in splits.items():

    print("\n" + "=" * 70)
    print(f"{split_name} RESULTS")
    print("=" * 70)

    y_true, y_pred, y_prob = predict_on_loader(loader)

    results = compute_metrics_with_ci(
        y_true,
        y_pred,
        y_prob,
        n_boot=NUM_BOOT,
        alpha=ALPHA
    )

    print("\nPoint Estimates and 95% Bootstrap CI")

    for metric, values in results.items():

        point = values["Point"]
        lower = values["Lower"]
        upper = values["Upper"]

        print(
            f"{metric:12s}: {point:.4f} "
            f"(95% CI: {lower:.4f} - {upper:.4f})"
        )

    all_results.append({
        "Split": split_name,
        **{
            metric: values["Point"]
            for metric, values in results.items()
        }
    })

    if split_name == "TEST":

        test_true = y_true
        test_pred = y_pred
        test_prob = y_prob

# ================================================================
# 15. CLASS-WISE TEST PERFORMANCE
# ================================================================

print("\n" + "=" * 70)
print("TEST CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    test_true,
    test_pred,
    labels=np.arange(num_classes),
    target_names=classes,
    digits=4,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

print(report_df)

report_df.to_csv(
    OUTPUT_DIR / "test_classification_report.csv"
)

# ================================================================
# 16. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    test_true,
    test_pred,
    labels=np.arange(num_classes)
)

cm_df = pd.DataFrame(
    cm,
    index=classes,
    columns=classes
)

cm_df.to_csv(
    OUTPUT_DIR / "test_confusion_matrix.csv"
)

print("\nConfusion matrix saved.")

# ================================================================
# 17. SAVE FINAL METRICS
# ================================================================

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    OUTPUT_DIR / "final_metrics.csv",
    index=False
)

print("\n" + "=" * 70)
print("FINAL PERFORMANCE SUMMARY")
print("=" * 70)

print(results_df.to_string(index=False))

# Save test predictions
pred_df = pd.DataFrame({
    "True_Label": test_true,
    "Predicted_Label": test_pred,
    "True_Class": [classes[i] for i in test_true],
    "Predicted_Class": [classes[i] for i in test_pred]
})

pred_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# Save probability predictions
prob_df = pd.DataFrame(
    test_prob,
    columns=[f"Prob_{c}" for c in classes]
)

prob_df.to_csv(
    OUTPUT_DIR / "test_probabilities.csv",
    index=False
)

print("\n" + "=" * 70)
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Best checkpoint:", best_model_path)
print("Results directory:", OUTPUT_DIR)

print("\nSaved files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)


CONVNEXT-TINY + GCN FISH CLASSIFICATION
Device: cuda

Scanning class folders recursively...

0: 954 images
1: 1000 images
2: 1000 images
3: 1000 images
4: 1000 images
5: 994 images
6: 1000 images
7: 1000 images

DATASET SUMMARY
Number of classes: 8
Total images: 7948

DATA SPLIT
--------------------------------------------------
Training:   5086 (63.99%)
Validation: 1272 (16.00%)
Testing:    1590 (20.01%)

ConvNeXt feature map: (768, 7, 7)


Epoch 1/10:   0%|          | 0/159 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/40 [00:00<?, ?it/s]


-------------------------------------------------------
Epoch: 1/10
Train Loss: 1.8634
Train Accuracy: 60.91%
Validation Accuracy: 86.87%
Best model checkpoint saved.


Epoch 2/10:   0%|          | 0/159 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/40 [00:00<?, ?it/s]

... (32 outputs left out)


Point Estimates and 95% Bootstrap CI
Accuracy    : 0.9969 (95% CI: 0.9937 - 0.9992)
Precision   : 0.9968 (95% CI: 0.9937 - 0.9992)
Recall      : 0.9969 (95% CI: 0.9938 - 0.9992)
F1          : 0.9968 (95% CI: 0.9936 - 0.9992)
AUC         : 1.0000 (95% CI: 0.9999 - 1.0000)

TEST RESULTS


Evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

Bootstrap confidence intervals:   0%|          | 0/2000 [00:00<?, ?it/s]


Point Estimates and 95% Bootstrap CI
Accuracy    : 0.9962 (95% CI: 0.9931 - 0.9987)
Precision   : 0.9962 (95% CI: 0.9931 - 0.9987)
Recall      : 0.9962 (95% CI: 0.9931 - 0.9988)
F1          : 0.9962 (95% CI: 0.9930 - 0.9987)
AUC         : 1.0000 (95% CI: 1.0000 - 1.0000)

TEST CLASSIFICATION REPORT
              precision    recall  f1-score      support
0              0.979487  1.000000  0.989637   191.000000
1              1.000000  1.000000  1.000000   200.000000
2              1.000000  1.000000  1.000000   200.000000
3              1.000000  0.990000  0.994975   200.000000
4              1.000000  1.000000  1.000000   200.000000
5              1.000000  0.984925  0.992405   199.000000
6              0.990099  1.000000  0.995025   200.000000
7              1.000000  0.995000  0.997494   200.000000
accuracy       0.996226  0.996226  0.996226     0.996226
macro avg      0.996198  0.996241  0.996192  1590.000000
weighted avg   0.996290  0.996226  0.996231  1590.000000

Confusion matr